# **Phase 2 — Modeling**

# **Imports**

In [1]:
import pandas as pd
import numpy as np
import re
import time
import ast
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# **Load Processed Data**

In [2]:
df = pd.read_csv('Processed_data.csv')

# Parse entities column from string to list
df['entities'] = df['entities'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

df.head()

,Text,Compartive,EN1,EN2,Sentiment,PreferedEntity,label,entities,brands,brand_count,has_comparison,text_length,word_count
0,انا الريدي ما وقفت ستريم ابل والسبوتيفاي حسابي...,non,NaN,NaN,NaN,NaN,0,[],['apple'],1,False,107,19
1,مقارنه بين هواوي p50 pro مع سامسونق s21 ultra ...,non,NaN,NaN,NaN,NaN,0,[],"['huawei', 'samsung']",2,True,74,13
2,والله اشتقتلك يلعن شكلك شفلنا سوني فايف بس بلا...,non,NaN,NaN,NaN,NaN,0,[],['sony'],1,False,59,13
3,هواوي انتهت خلاص,non,NaN,NaN,NaN,NaN,0,[],['huawei'],1,False,16,3
4,المهم سوني افضل من اكس بوكس خلاص انهيتها plays...,com,سوني,اكس بوكس,p,سوني,1,"[سوني, اكس بوكس]",['sony'],1,True,53,9


# **Train / Validation / Test Split**

In [3]:
X = df["Text"]
y = df["label"]

# 70% Train
# 15% Validation
# 15% Test

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"Train : {len(X_train)}")
print(f"Val   : {len(X_val)}")
print(f"Test  : {len(X_test)}")

Train : 3378
Val   : 724
Test  : 724


# **Evaluation Helper**

In [4]:
def evaluate(
    name,
    y_true_train,
    y_pred_train,
    y_true_val,
    y_pred_val,
    y_true_test,
    y_pred_test,
    elapsed
):

    def metrics(y_true, y_pred):
        return {
            "acc": accuracy_score(y_true, y_pred),
            "p": precision_score(y_true, y_pred, zero_division=0),
            "r": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)
        }

    train = metrics(y_true_train, y_pred_train)
    val = metrics(y_true_val, y_pred_val)
    test = metrics(y_true_test, y_pred_test)

    ms = elapsed * 1000 / len(y_true_test)

    print(f"\n── {name} ──")
    print(f' {"Split":<6} {"Acc":>7} {"Precision":>10} {"Recall":>8} {"F1":>8}')
    print(f' {"─"*43}')

    print(
        f' {"Train":<6} '
        f'{train["acc"]:>7.4f} '
        f'{train["p"]:>10.4f} '
        f'{train["r"]:>8.4f} '
        f'{train["f1"]:>8.4f}'
    )

    print(
        f' {"Val":<6} '
        f'{val["acc"]:>7.4f} '
        f'{val["p"]:>10.4f} '
        f'{val["r"]:>8.4f} '
        f'{val["f1"]:>8.4f}'
    )

    print(
        f' {"Test":<6} '
        f'{test["acc"]:>7.4f} '
        f'{test["p"]:>10.4f} '
        f'{test["r"]:>8.4f} '
        f'{test["f1"]:>8.4f}'
    )

    print(f" Inference : {ms:.2f} ms/sample")

    return {
        "model": name,
        "test_acc": round(test["acc"], 4),
        "test_precision": round(test["p"], 4),
        "test_recall": round(test["r"], 4),
        "test_f1": round(test["f1"], 4),
        "ms_per_sample": round(ms, 2)
    }

# **Approach 1 — Classical ML (TF-IDF + SVM)**

In [5]:
# ===================================================
#  Imports
# ===================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

In [6]:
svm_pipeline = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 3),
            max_features=30000,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True )
    ),

    (
        "clf",
        LinearSVC(
            C=0.1,
            max_iter=3000 )
    )

])

In [7]:
svm_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=30000, min_df=2,
                                 ngram_range=(1, 3), sublinear_tf=True)),
                ('clf', LinearSVC(C=0.1, max_iter=3000))])

In [8]:
y_pred_train_svm = svm_pipeline.predict(
    X_train
)

y_pred_val_svm = svm_pipeline.predict(
    X_val
)

start = time.time()

y_pred_test_svm = svm_pipeline.predict(
    X_test
)

elapsed_svm = time.time() - start

In [9]:
results_svm = evaluate(
    "Classical ML — TF-IDF + SVM",

    y_train,
    y_pred_train_svm,

    y_val,
    y_pred_val_svm,

    y_test,
    y_pred_test_svm,

    elapsed_svm
)


── Classical ML — TF-IDF + SVM ──
 Split      Acc  Precision   Recall       F1
 ───────────────────────────────────────────
 Train   0.9340     0.8807   0.9903   0.9323
 Val     0.9130     0.8512   0.9819   0.9119
 Test    0.9047     0.8380   0.9819   0.9043
 Inference : 0.07 ms/sample


In [10]:
cm = confusion_matrix(
    y_test,
    y_pred_test_svm
)

print(cm)

[[329  63]
 [  6 326]]


In [11]:
results_svm

{'model': 'Classical ML — TF-IDF + SVM',
 'test_acc': 0.9047,
 'test_precision': 0.838,
 'test_recall': 0.9819,
 'test_f1': 0.9043,
 'ms_per_sample': 0.07}

# **Approach 2 — Deep Learning (BiLSTM)**

In [12]:
# ===================================================
#  Imports
# ===================================================

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [13]:
MAX_WORDS = 20000
MAX_LEN = 100

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

## Text → Sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [14]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [15]:
# ===================================================
# Download FastText Arabic Embeddings (Run Once)
# ===================================================

import os

if not os.path.exists("cc.ar.300.vec"):

    !wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.vec.gz

    !gunzip -f cc.ar.300.vec.gz

print(
    os.path.exists("cc.ar.300.vec")
)

True


In [16]:
EMBEDDING_DIM = 300

embeddings_index = {}

with open(
    "cc.ar.300.vec",
    encoding="utf-8"
) as f:

    next(f)

    for line in f:

        values = line.rstrip().split(" ")

        word = values[0]

        vector = np.asarray(
            values[1:],
            dtype="float32"
        )

        if len(vector) == EMBEDDING_DIM:
            embeddings_index[word] = vector

print(
    "Loaded",
    len(embeddings_index),
    "word vectors"
)

Loaded 2000000 word vectors


In [17]:
embedding_matrix = np.zeros(
    (MAX_WORDS, EMBEDDING_DIM)
)

word_index = tokenizer.word_index

hits = 0
misses = 0

for word, idx in word_index.items():

    if idx >= MAX_WORDS:
        continue

    vector = embeddings_index.get(word)

    if vector is not None:

        embedding_matrix[idx] = vector
        hits += 1

    else:

        misses += 1

print("Found :", hits)
print("Missed:", misses)

coverage = hits / (hits + misses)

print(
    f"Coverage = {coverage:.2%}"
)

Found : 10958
Missed: 1449
Coverage = 88.32%


In [18]:
# ===================================================
#  Imports
# ===================================================

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

In [19]:
bilstm_model = Sequential([

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=False
    ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=True,
            dropout=0.2
        )
    ),

    Bidirectional(
        LSTM(
            64,
            dropout=0.2
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )

])

In [20]:
bilstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,000,000 (22.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,000,000 (22.89 MB)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [22]:
print("model" in globals())

False


In [23]:
## Step 9 — Compile

bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,000,000 (22.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,000,000 (22.89 MB)

In [24]:
# =======================
# Imports
# =======================
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [25]:
history = bilstm_model.fit(

    X_train_pad,
    y_train,

    validation_data=(
        X_val_pad,
        y_val
    ),

    epochs=15,
    batch_size=16,

    callbacks=[early_stop],

    verbose=1
)

Epoch 1/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 16s 37ms/step - accuracy: 0.8179 - loss: 0.4163 - val_accuracy: 0.9116 - val_loss: 0.2499
Epoch 2/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8878 - loss: 0.2835 - val_accuracy: 0.9157 - val_loss: 0.2456
Epoch 3/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.8934 - loss: 0.2761 - val_accuracy: 0.9130 - val_loss: 0.2373
Epoch 4/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8961 - loss: 0.2508 - val_accuracy: 0.9130 - val_loss: 0.2272
Epoch 5/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.8976 - loss: 0.2444 - val_accuracy: 0.9157 - val_loss: 0.2384
Epoch 6/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8982 - loss: 0.2460 - val_accuracy: 0.9088 - val_loss: 0.2622
Epoch 7/15
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9020 - loss: 0.2488 - val_accuracy: 0.9144 - val_loss: 0.2384


In [26]:
y_pred_train_bilstm = (
    bilstm_model.predict(
        X_train_pad,
        verbose=0
    ) > 0.5
).astype(int)

y_pred_val_bilstm = (
    bilstm_model.predict(
        X_val_pad,
        verbose=0
    ) > 0.5
).astype(int)

start = time.time()

y_pred_test_bilstm = (
    bilstm_model.predict(
        X_test_pad,
        verbose=0
    ) > 0.5
).astype(int)

elapsed_bilstm = (
    time.time() - start
)

In [27]:
results_bilstm = evaluate(

    "Deep Learning — BiLSTM + FastText",

    y_train,
    y_pred_train_bilstm,

    y_val,
    y_pred_val_bilstm,

    y_test,
    y_pred_test_bilstm,

    elapsed_bilstm
)


── Deep Learning — BiLSTM + FastText ──
 Split      Acc  Precision   Recall       F1
 ───────────────────────────────────────────
 Train   0.9005     0.8225   0.9987   0.9021
 Val     0.9130     0.8422   0.9970   0.9131
 Test    0.9019     0.8238   1.0000   0.9034
 Inference : 0.98 ms/sample


In [28]:
cm = confusion_matrix(
    y_test,
    y_pred_test_bilstm
)

print(cm)

[[321  71]
 [  0 332]]


In [29]:
results_bilstm

{'model': 'Deep Learning — BiLSTM + FastText',
 'test_acc': 0.9019,
 'test_precision': 0.8238,
 'test_recall': 1.0,
 'test_f1': 0.9034,
 'ms_per_sample': 0.98}

# **Approach 3 — Transformer (AraBERT Fine-tuning)**

In [30]:
!pip install transformers -q

In [31]:
import torch
import numpy as np
import time

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from tqdm import tqdm

In [32]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"

hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

arabert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

arabert_model = arabert_model.to(device)

print(device)

cuda


In [34]:
class ArabicDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len
    ):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].squeeze(),

            "attention_mask":
                encoding["attention_mask"].squeeze(),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [35]:
MAX_LEN = 128

train_dataset = ArabicDataset(
    X_train,
    y_train,
    hf_tokenizer,
    MAX_LEN
)

val_dataset = ArabicDataset(
    X_val,
    y_val,
    hf_tokenizer,
    MAX_LEN
)

test_dataset = ArabicDataset(
    X_test,
    y_test,
    hf_tokenizer,
    MAX_LEN
)

In [36]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

In [37]:
optimizer = AdamW(
    arabert_model.parameters(),
    lr=2e-5
)

In [38]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [39]:
def evaluate_epoch(model, loader):

    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

            true_labels.extend(
                labels.cpu().numpy()
            )

    acc = accuracy_score(
        true_labels,
        predictions
    )

    p = precision_score(
        true_labels,
        predictions
    )

    r = recall_score(
        true_labels,
        predictions
    )

    f1 = f1_score(
        true_labels,
        predictions
    )

    return acc, p, r, f1

In [40]:
EPOCHS = 2

for epoch in range(EPOCHS):

    train_loss = train_epoch(
        arabert_model,
        train_loader
    )

    val_acc, val_p, val_r, val_f1 = evaluate_epoch(
        arabert_model,
        val_loader
    )

    print()
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Acc : {val_acc:.4f}")
    print(f"Val F1  : {val_f1:.4f}")

100%|██████████| 212/212 [01:18<00:00,  2.71it/s]



Epoch 1/2
Train Loss: 0.3106
Val Acc : 0.9157
Val F1  : 0.9159


100%|██████████| 212/212 [01:18<00:00,  2.69it/s]



Epoch 2/2
Train Loss: 0.2025
Val Acc : 0.9296
Val F1  : 0.9268


In [41]:
start = time.time()

test_acc, test_p, test_r, test_f1 = evaluate_epoch(
    arabert_model,
    test_loader
)

elapsed = time.time() - start

ms = elapsed * 1000 / len(y_test)

print("\n── Transformer — AraBERT ──")

print(f"Test Acc       : {test_acc:.4f}")
print(f"Test Precision : {test_p:.4f}")
print(f"Test Recall    : {test_r:.4f}")
print(f"Test F1        : {test_f1:.4f}")
print(f"Inference      : {ms:.2f} ms/sample")


── Transformer — AraBERT ──
Test Acc       : 0.9102
Test Precision : 0.8638
Test Recall    : 0.9548
Test F1        : 0.9070
Inference      : 8.82 ms/sample


In [42]:
arabert_model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        outputs = arabert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        true_labels.extend(
            batch["labels"].numpy()
        )

cm = confusion_matrix(
    true_labels,
    predictions
)

print(cm)

[[342  50]
 [ 15 317]]


In [43]:
results_arabert = {
    "model": "Transformer — AraBERT",
    "test_acc": round(test_acc, 4),
    "test_precision": round(test_p, 4),
    "test_recall": round(test_r, 4),
    "test_f1": round(test_f1, 4),
    "ms_per_sample": round(ms, 2)
}

results_arabert

{'model': 'Transformer — AraBERT',
 'test_acc': 0.9102,
 'test_precision': 0.8638,
 'test_recall': 0.9548,
 'test_f1': 0.907,
 'ms_per_sample': 8.82}

# **Approach 4 — LLM**

In [44]:
!pip install openai -q

import time

from openai import OpenAI

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [45]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_API_KEY"
)

In [46]:
def build_prompt(tweet):

    return f"""
You are an expert Arabic Comparative Opinion Mining classifier.

Task:
Classify the tweet.

Return ONLY:
1
or
0

Label = 1 ONLY IF:
- The author expresses a comparative opinion.
- The author clearly prefers one entity over another.
- There is a clear winner and loser.

Label = 0 IF:
- It is a question.
- It asks for advice.
- It asks which is better.
- It discusses a comparison.
- It reports someone else's comparison.
- It mentions multiple brands without preference.
- It is news or information.
- No clear comparative opinion from the author.

Examples:

Tweet:
سامسونج افضل من ابل بمراحل
Answer:
1

Tweet:
ايفون افضل من اي هاتف اندرويد
Answer:
1

Tweet:
سوني تتفوق على مايكروسوفت
Answer:
1

Tweet:
اخ محمد تنصحني بالابتوب ولا الايباد افضل؟
Answer:
0

Tweet:
هل ايفون افضل من جالكسي؟
Answer:
0

Tweet:
انت تريد تقنعنا ان ايفون افضل من جالكسي
Answer:
0

Tweet:
اتش بي افضل من سامسونج وبكم سعرها؟
Answer:
0

Tweet:
ابل اكبر مستهلك لخدمات جوجل السحابية
Answer:
0

Tweet:
سامسونج وابل شركتان كبيرتان
Answer:
0

Tweet:
ودي اشوف ليه الناس تستخدم ايفون اكثر من هواوي
Answer:
0

Tweet:
{tweet}

Answer:
"""

In [47]:
def predict_qwen(tweet):

    prompt = build_prompt(tweet)

    response = client.chat.completions.create(
        model="qwen/qwen-2.5-7b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=3
    )

    return response.choices[0].message.content.strip()

In [48]:
sample = X_test.iloc[0]

print(sample)

print(
    predict_qwen(sample)
)

والله ان سامسونج افضل من ابل الف مرا
1


In [49]:
predictions = []

start = time.time()

for text in X_test:

    pred = predict_qwen(text)

    pred = 1 if str(pred).startswith("1") else 0

    predictions.append(pred)

elapsed = time.time() - start

In [50]:
acc = accuracy_score(
    y_test,
    predictions
)

p = precision_score(
    y_test,
    predictions
)

r = recall_score(
    y_test,
    predictions
)

f1 = f1_score(
    y_test,
    predictions
)

ms = elapsed * 1000 / len(y_test)

In [51]:
print("\n── LLM — Qwen Few-Shot ──")

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {p:.4f}")
print(f"Recall    : {r:.4f}")
print(f"F1        : {f1:.4f}")
print(f"Inference : {ms:.2f} ms/sample")


── LLM — Qwen Few-Shot ──
Accuracy  : 0.8191
Precision : 0.8131
Recall    : 0.7861
F1        : 0.7994
Inference : 1336.23 ms/sample


In [52]:
cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

[[332  60]
 [ 71 261]]


In [53]:
# **Approach 4 — LLM**

In [54]:
results_qwen = {
    "model": "LLM — Qwen Few-Shot",
    "test_acc": round(acc, 4),
    "test_precision": round(p, 4),
    "test_recall": round(r, 4),
    "test_f1": round(f1, 4),
    "ms_per_sample": round(ms, 2)
}

results_qwen

{'model': 'LLM — Qwen Few-Shot',
 'test_acc': 0.8191,
 'test_precision': 0.8131,
 'test_recall': 0.7861,
 'test_f1': 0.7994,
 'ms_per_sample': 1336.23}

# **Phase 3 — Evaluation**

In [55]:
comparison = pd.DataFrame([
    results_svm,
    results_bilstm,
    results_arabert,
    results_qwen
])

comparison

,model,test_acc,test_precision,test_recall,test_f1,ms_per_sample
0,Classical ML — TF-IDF + SVM,0.9047,0.8380,0.9819,0.9043,0.07
1,Deep Learning — BiLSTM + FastText,0.9019,0.8238,1.0000,0.9034,0.98
2,Transformer — AraBERT,0.9102,0.8638,0.9548,0.9070,8.82
3,LLM — Qwen Few-Shot,0.8191,0.8131,0.7861,0.7994,1336.23


In [56]:
ner_df = df.copy()

ner_df = ner_df[
    ner_df["entities"].apply(len) > 0
].reset_index(drop=True)

print("NER Samples:", len(ner_df))

ner_df[
    ["Text", "entities"]
].head()

NER Samples: 2218


,Text,entities
0,المهم سوني افضل من اكس بوكس خلاص انهيتها plays...,"[سوني, اكس بوكس]"
1,يا شباب ترا نفسها بس سامسونج تدلع شاشاتها وشاش...,"[سامسونج, BOE]"
2,انا اشوف سوني افضل من اكسبوكس و البيسي افضل من...,"[سوني, اكسبوكس]"
3,جربت التويتر في سامسونج افضل من ايفون بي ٨٢٧٣٦...,"[سامسونج, ايفون]"
4,ابل افضل من ناحيه تنزيل الاغاني والموسوعه حقته...,"[ابل, انغامي]"


In [57]:
def text_to_bio(text, entities):

    tokens = text.split()

    labels = ["O"] * len(tokens)

    for entity in entities:

        entity_tokens = str(entity).split()

        n = len(entity_tokens)

        for i in range(len(tokens) - n + 1):

            if tokens[i:i+n] == entity_tokens:

                labels[i] = "B-BRAND"

                for j in range(1, n):

                    labels[i+j] = "I-BRAND"

    return tokens, labels


ner_tokens = []
ner_labels = []

for _, row in ner_df.iterrows():

    tokens, labels = text_to_bio(
        row["Text"],
        row["entities"]
    )

    ner_tokens.append(tokens)
    ner_labels.append(labels)

In [58]:
label_list = [
    "O",
    "B-BRAND",
    "I-BRAND"
]

label2id = {
    label: idx
    for idx, label in enumerate(label_list)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print(label2id)

{'O': 0, 'B-BRAND': 1, 'I-BRAND': 2}


In [59]:
from sklearn.model_selection import train_test_split

X_train_tokens, X_temp_tokens, y_train_labels, y_temp_labels = train_test_split(
    ner_tokens,
    ner_labels,
    test_size=0.30,
    random_state=42
)

X_val_tokens, X_test_tokens, y_val_labels, y_test_labels = train_test_split(
    X_temp_tokens,
    y_temp_labels,
    test_size=0.50,
    random_state=42
)

print("Train:", len(X_train_tokens))
print("Val  :", len(X_val_tokens))
print("Test :", len(X_test_tokens))

Train: 1552
Val  : 333
Test : 333


In [60]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

MODEL_NAME = "aubmindlab/bert-base-arabertv02"

hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

In [61]:
from torch.utils.data import Dataset

class NERDataset(Dataset):

    def __init__(
        self,
        tokens,
        labels,
        tokenizer,
        max_len=128
    ):
        self.tokens = tokens
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):

        words = self.tokens[idx]

        word_labels = [
            label2id[x]
            for x in self.labels[idx]
        ]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        label_ids = []

        previous_word = None

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word:

                label_ids.append(
                    word_labels[word_idx]
                )

            else:

                label_ids.append(
                    word_labels[word_idx]
                )

            previous_word = word_idx

        return {
            "input_ids":
                encoding["input_ids"].squeeze(),

            "attention_mask":
                encoding["attention_mask"].squeeze(),

            "labels":
                torch.tensor(label_ids)
        }

In [62]:
from torch.utils.data import DataLoader

train_dataset = NERDataset(
    X_train_tokens,
    y_train_labels,
    hf_tokenizer
)

val_dataset = NERDataset(
    X_val_tokens,
    y_val_labels,
    hf_tokenizer
)

test_dataset = NERDataset(
    X_test_tokens,
    y_test_labels,
    hf_tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

In [64]:
import torch
from torch.optim import AdamW

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

ner_model = ner_model.to(device)

optimizer = AdamW(
    ner_model.parameters(),
    lr=2e-5
)

print(device)

cuda


In [65]:
from tqdm import tqdm

def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [66]:
def evaluate_token_accuracy(
    model,
    loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(
                outputs.logits,
                dim=2
            )

            mask = labels != -100

            correct += (
                (preds == labels)[mask]
            ).sum().item()

            total += mask.sum().item()

    return correct / total

In [67]:
EPOCHS = 3

for epoch in range(EPOCHS):

    train_loss = train_epoch(
        ner_model,
        train_loader
    )

    val_acc = evaluate_token_accuracy(
        ner_model,
        val_loader
    )

    print()
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Loss      : {train_loss:.4f}")
    print(f"Token Acc : {val_acc:.4f}")

100%|██████████| 97/97 [00:35<00:00,  2.70it/s]



Epoch 1/3
Loss      : 0.2152
Token Acc : 0.9579


100%|██████████| 97/97 [00:36<00:00,  2.64it/s]



Epoch 2/3
Loss      : 0.1092
Token Acc : 0.9607


100%|██████████| 97/97 [00:36<00:00,  2.63it/s]



Epoch 3/3
Loss      : 0.0759
Token Acc : 0.9603


In [69]:
!pip install seqeval -q

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

true_predictions = []
true_labels = []

ner_model.eval()

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = ner_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=2
        )

        preds = preds.cpu().numpy()
        labels = labels.cpu().numpy()

        for pred, label in zip(preds, labels):

            pred_tags = []
            true_tags = []

            for p, l in zip(pred, label):

                if l != -100:

                    pred_tags.append(
                        id2label[p]
                    )

                    true_tags.append(
                        id2label[l]
                    )

            true_predictions.append(pred_tags)
            true_labels.append(true_tags)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [70]:
ner_precision = precision_score(
    true_labels,
    true_predictions
)

ner_recall = recall_score(
    true_labels,
    true_predictions
)

ner_f1 = f1_score(
    true_labels,
    true_predictions
)

print("\n── NER Evaluation ──")
print(f"Precision : {ner_precision:.4f}")
print(f"Recall    : {ner_recall:.4f}")
print(f"F1 Score  : {ner_f1:.4f}")

print(
    classification_report(
        true_labels,
        true_predictions
    )
)


── NER Evaluation ──
Precision : 0.8457
Recall    : 0.8570
F1 Score  : 0.8513
              precision    recall  f1-score   support

       BRAND       0.85      0.86      0.85      1049

   micro avg       0.85      0.86      0.85      1049
   macro avg       0.85      0.86      0.85      1049
weighted avg       0.85      0.86      0.85      1049



# **Phase 4 —  Engineering & Deployment**

In [71]:
arabert_model.save_pretrained(
    "comparison_model"
)

hf_tokenizer.save_pretrained(
    "comparison_model"
)


ner_model.save_pretrained(
    "ner_model"
)

hf_tokenizer.save_pretrained(
    "ner_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('ner_model/tokenizer_config.json', 'ner_model/tokenizer.json')

In [72]:
import os

print(
    os.listdir("comparison_model")
)

print(
    os.listdir("ner_model")
)

['model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']
['model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


In [82]:
!pip install fastapi uvicorn transformers torch -q

In [83]:
!nohup uvicorn app:app \
    --host 0.0.0.0 \
    --port 8000 \
    > server.log 2>&1 &

In [84]:
import requests

response = requests.get(
    "http://127.0.0.1:8000"
)

print(
    response.json()
)

{'status': 'running'}


## API Testing

After deploying the FastAPI application, the following tests were performed to verify that the API was functioning correctly.

In [96]:
import requests

response = requests.get(
    "http://127.0.0.1:8000"
)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 200
{'status': 'running'}


In [97]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={
        "text": "سامسونج أفضل من ايفون بمراحل"
    }
)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 200
{'classification': 1, 'entities': ['سامسونج', 'ايفون']}


In [98]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={
        "text": "الهاتف جميل جدا"
    }
)

print(response.json())

{'classification': 0, 'entities': []}


## Deployment Verification

The API was successfully deployed using FastAPI and tested through HTTP requests. Both the health check endpoint and prediction endpoint returned valid responses, confirming successful deployment and model integration.